# M4 Evaluation：四组配置与失败归因

固定数据、Agent 策略、top_k 等无关变量，只改变启用的记忆类型。

报告 routing accuracy、writing compliance、required-memory recall，并保存逐题结果。

In [ ]:
import json

from email_memory_lab import (
    AgentConfig,
    LocalStore,
    evaluate_config,
    find_project_root,
    load_labeled_emails,
    seed_memories,
    start_run,
)

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
cases = load_labeled_emails(DATA / "emails.json")
store = LocalStore()
seed_memories(store, DATA / "memories.json", "user-a")
run = start_run(PROJECT_ROOT, "04-evaluation", {"memory": "all"})

In [ ]:
configs = [
    AgentConfig(),
    AgentConfig(use_semantic=True),
    AgentConfig(use_episodic=True),
    AgentConfig(use_procedural=True),
    AgentConfig(use_semantic=True, use_episodic=True, use_procedural=True),
]

results = {config.label: evaluate_config(cases, store, "user-a", config) for config in configs}
print(" | ".join(results))
for label, result in results.items():
    summary = result["summary"]
    print(
        f"{label}: routing {summary['routing_accuracy']['passed']}/{summary['routing_accuracy']['total']}, "
        f"writing {summary['writing_compliance']['passed']}/{summary['writing_compliance']['total']}, "
        f"memory {summary['required_memory_recall']['passed']}/{summary['required_memory_recall']['total']}"
    )

## 相对 baseline 的差值

这里只做同题对照，不把小样本差值当成统计显著提升。

In [ ]:
baseline = results["baseline"]
for label, result in results.items():
    deltas = {}
    for metric in ("routing_accuracy", "writing_compliance", "required_memory_recall"):
        before = baseline["summary"][metric]["value"]
        after = result["summary"][metric]["value"]
        deltas[metric] = round((after or 0.0) - (before or 0.0), 4)
    print(label, deltas)

In [ ]:
full = results["semantic+episodic+procedural"]
print("full failures:", [row["email_id"] for row in full["failures"]])
for row in full["cases"]:
    print(
        f"{row['email_id']}: {row['predicted_action']} "
        f"(action_ok={row['action_ok']}, writing_ok={row['writing_ok']}, memory={row['used_memory']})"
    )

## 报告时必须说明

- 只有 10 封邮件，这只验证机制，不是统计结论；
- Agent 使用确定性策略，不代表相同结果会出现在 LLM 上；
- full memory 的改善分别由 semantic、episodic、procedural 哪一例贡献；
- 保存 manifest、逐题结果和失败案例，便于复核。

In [ ]:
path = run.save("evaluation-report.json", results)
print(f"Saved: {path}")
print(f"Manifest: {run.directory / 'manifest.json'}")